# machine-learning_003

> **Auto-generated documentation header.** Fill the `TODO` fields after review.
> Source notebook: `Notebook1 (118).ipynb`

| Field | Value |
|---|---|
| Section | `machine_learning` |
| Market | `unknown` |
| Trading style | `unknown` |
| Timeframe | `unknown` |
| Code cells | 13 |
| Detected functions | classify_trade, classify_trade, sharpe |

### Research documentation
- **Hypothesis:** _TODO_
- **Alpha source:** _TODO_
- **Indicators / features:** _TODO_
- **Entry logic:** _TODO_
- **Exit logic (SL/TP):** _TODO_
- **Risk management:** _TODO_
- **Assumptions & limitations:** _TODO_
- **Performance (if available):** _TODO_

> Shared utilities live in `src/qresearch/` — prefer importing `jma_signals`, `backtest_trades`,
> `calculate_metrics`, `resample_ohlcv` instead of re-implementing them here.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

df = pd.read_parquet('/content/drive/MyDrive/5min_data_ADANIENT.parquet')
df

In [ ]:
df1 = pd.read_csv('/content/drive/MyDrive/tradebook_ADANIENT_2024.csv')
df1

In [ ]:
df.columns

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Make sure Date is datetime and sort the data
df['Date'] = pd.to_datetime(df['Date'])
df['year'] = df['Date'].dt.year
df = df.sort_values(by=['Ticker', 'Date'])

# -------------------------------
# 1. SAMX Distribution Shift Over Years
# -------------------------------
plt.figure(figsize=(12, 6))
for y in [2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]:
    sns.kdeplot(df[df['year'] == y]['SAMX'], label=str(y))
plt.title('SAMX Signal Distribution Over Years')
plt.xlabel('SAMX')
plt.ylabel('Density')
plt.legend()
plt.grid(True)
plt.show()

# -------------------------------
# 2. Average SAMX vs. Realized Return
# -------------------------------
df['future_return'] = df.groupby('Ticker')['close'].shift(-1) / df['close'] - 1

# Bin SAMX values
df['SAMX_bin'] = pd.qcut(df['SAMX'], q=10, duplicates='drop')

plt.figure(figsize=(14, 6))
sns.boxplot(x='SAMX_bin', y='future_return', data=df)
plt.xticks(rotation=45)
plt.title('Future Return by SAMX Quantile')
plt.ylabel('Next Candle Return')
plt.grid(True)
plt.show()

# -------------------------------
# 3. Win Rate Per Year (for long and short)
# -------------------------------
def classify_trade(row):
    if row['signal'] == 1:
        return 1 if row['future_return'] > 0 else 0
    elif row['signal'] == 2:
        return 1 if row['future_return'] < 0 else 0
    return np.nan

df['win'] = df.apply(classify_trade, axis=1)

winrate_by_year = df.groupby('year')['win'].mean()

plt.figure(figsize=(10, 5))
winrate_by_year.plot(kind='bar', color='teal')
plt.axhline(0.5, linestyle='--', color='red', label='50% threshold')
plt.title('Win Rate by Year')
plt.ylabel('Win Rate')
plt.legend()
plt.grid(True)
plt.show()

# -------------------------------
# 4. Trades Taken Per Year
# -------------------------------
trades_by_year = df[df['signal'].isin([1, 2])].groupby('year').size()

plt.figure(figsize=(10, 5))
trades_by_year.plot(kind='bar', color='orange')
plt.title('Trades Executed Per Year')
plt.ylabel('Number of Trades')
plt.grid(True)
plt.show()

# -------------------------------
# 5. Avg PnL per Trade by Year (if available)
# -------------------------------
if 'PnL' in df.columns:
    avg_pnl = df.groupby('year')['PnL'].mean()
    plt.figure(figsize=(10, 5))
    avg_pnl.plot(kind='bar', color='purple')
    plt.title('Average PnL Per Trade by Year')
    plt.ylabel('Average PnL')
    plt.grid(True)
    plt.show()

# -------------------------------
# 6. Signal Strength Over Time
# -------------------------------
plt.figure(figsize=(14, 6))
df.groupby('year')['SAMX'].mean().plot(marker='o', label='SAMX')
df.groupby('year')['PVS'].mean().plot(marker='x', label='PVS')
df.groupby('year')['VMS'].mean().plot(marker='s', label='VMS')
plt.title('Signal Strength Averages Over Years')
plt.ylabel('Signal Value')
plt.legend()
plt.grid(True)
plt.show()


In [ ]:
import pandas as pd
import plotly.graph_objects as go
import plotly.express as px

# Assuming df1 and df are loaded into DataFrames

# ------------------------------
# 1. Cumulative PnL over Time (from df1)
# ------------------------------
df1['Entry Time'] = pd.to_datetime(df1['Entry Time'])
df1['Exit Time'] = pd.to_datetime(df1['Exit Time'])

# Plot Cumulative PnL
fig = go.Figure()
fig.add_trace(go.Scatter(x=df1['Exit Time'], y=df1['Cumulative PnL'], mode='lines', name='Cumulative PnL'))
fig.update_layout(title='Cumulative PnL Over Time',
                  xaxis_title='Time',
                  yaxis_title='Cumulative PnL',
                  template='plotly_dark')
fig.show()

# ------------------------------
# 2. Win Rate by Exit Reason
# ------------------------------
win_rate_by_exit_reason = df1.groupby('Exit Reason')['PnL'].apply(lambda x: (x > 0).mean())

# Plot win rate by exit reason
fig = go.Figure()
fig.add_trace(go.Bar(x=win_rate_by_exit_reason.index, y=win_rate_by_exit_reason.values, marker_color='teal'))
fig.update_layout(title='Win Rate by Exit Reason',
                  xaxis_title='Exit Reason',
                  yaxis_title='Win Rate',
                  template='plotly_dark')
fig.show()

# ------------------------------
# 3. PnL Distribution by Direction (Long/Short)
# ------------------------------
fig = px.box(df1, x='Direction', y='PnL', color='Direction', title='PnL Distribution by Direction (Long vs Short)',
             labels={'PnL': 'PnL', 'Direction': 'Direction'})
fig.update_layout(template='plotly_dark')
fig.show()

# ------------------------------
# 4. PnL Distribution by Year
# ------------------------------
df1['Year'] = df1['Exit Time'].dt.year

# Plot PnL Distribution by Year
fig = px.box(df1, x='Year', y='PnL', title='PnL Distribution by Year', labels={'PnL': 'PnL'})
fig.update_layout(template='plotly_dark')
fig.show()

# ------------------------------
# 5. Trade Frequency Over Time
# ------------------------------
trades_by_year = df1.groupby('Year').size()

# Plot trade frequency over time
fig = go.Figure()
fig.add_trace(go.Bar(x=trades_by_year.index, y=trades_by_year.values, marker_color='orange'))
fig.update_layout(title='Trade Frequency Over Time',
                  xaxis_title='Year',
                  yaxis_title='Number of Trades',
                  template='plotly_dark')
fig.show()

# ------------------------------
# 6. Exit Time Analysis
# ------------------------------
df1['Exit Hour'] = df1['Exit Time'].dt.hour

# Plot PnL by Exit Hour
fig = px.box(df1, x='Exit Hour', y='PnL', title='PnL Distribution by Exit Hour', labels={'PnL': 'PnL', 'Exit Hour': 'Exit Hour'})
fig.update_layout(template='plotly_dark')
fig.show()

# ------------------------------
# Additional Insights from df (8 Years Data)
# ------------------------------
# 1. SAMX Signal Distribution Shift Over Years
fig = go.Figure()
for y in [2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]:
    fig.add_trace(go.Histogram(x=df[df['Date'].dt.year == y]['SAMX'], name=str(y), opacity=0.75))
fig.update_layout(title='SAMX Signal Distribution Over Years',
                  xaxis_title='SAMX',
                  yaxis_title='Density',
                  barmode='overlay',
                  template='plotly_dark')
fig.show()

# 2. Average SAMX vs. Realized Return (future return)
df['future_return'] = df.groupby('Ticker')['close'].shift(-1) / df['close'] - 1
df['SAMX_bin'] = pd.qcut(df['SAMX'], q=10, duplicates='drop')

fig = px.box(df, x='SAMX_bin', y='future_return', title='Future Return by SAMX Quantile', labels={'future_return': 'Next Candle Return'})
fig.update_layout(template='plotly_dark')
fig.show()

# 3. Win Rate Per Year (for long and short)
def classify_trade(row):
    if row['signal'] == 1:
        return 1 if row['future_return'] > 0 else 0
    elif row['signal'] == 2:
        return 1 if row['future_return'] < 0 else 0
    return np.nan

df['win'] = df.apply(classify_trade, axis=1)

winrate_by_year = df.groupby('Date').apply(lambda x: (x['win'] == 1).mean())

fig = go.Figure()
fig.add_trace(go.Bar(x=winrate_by_year.index, y=winrate_by_year.values, marker_color='teal'))
fig.add_trace(go.Scatter(x=winrate_by_year.index, y=[0.5] * len(winrate_by_year), mode='lines', name='50% threshold', line=dict(dash='dash', color='red')))
fig.update_layout(title='Win Rate by Year',
                  xaxis_title='Year',
                  yaxis_title='Win Rate',
                  template='plotly_dark')
fig.show()

# 4. Trades Taken Per Year
trades_by_year = df[df['signal'].isin([1, 2])].groupby('Date').size()

fig = go.Figure()
fig.add_trace(go.Bar(x=trades_by_year.index, y=trades_by_year.values, marker_color='orange'))
fig.update_layout(title='Trades Executed Per Year',
                  xaxis_title='Year',
                  yaxis_title='Number of Trades',
                  template='plotly_dark')
fig.show()

# 5. Avg PnL per Trade by Year (if available)
if 'PnL' in df.columns:
    avg_pnl = df.groupby(df['Date'].dt.year)['PnL'].mean()

    fig = go.Figure()
    fig.add_trace(go.Bar(x=avg_pnl.index, y=avg_pnl.values, marker_color='purple'))
    fig.update_layout(title='Average PnL Per Trade by Year',
                      xaxis_title='Year',
                      yaxis_title='Average PnL',
                      template='plotly_dark')
    fig.show()

# 6. Signal Strength Over Time (for SAMX, PVS, VMS)
fig = go.Figure()
fig.add_trace(go.Scatter(x=df.groupby(df['Date'].dt.year)['SAMX'].mean().index,
                         y=df.groupby(df['Date'].dt.year)['SAMX'].mean().values, mode='lines', name='SAMX'))
fig.add_trace(go.Scatter(x=df.groupby(df['Date'].dt.year)['PVS'].mean().index,
                         y=df.groupby(df['Date'].dt.year)['PVS'].mean().values, mode='lines', name='PVS'))
fig.add_trace(go.Scatter(x=df.groupby(df['Date'].dt.year)['VMS'].mean().index,
                         y=df.groupby(df['Date'].dt.year)['VMS'].mean().values, mode='lines', name='VMS'))

fig.update_layout(title='Signal Strength Averages Over Years',
                  xaxis_title='Year',
                  yaxis_title='Signal Value',
                  template='plotly_dark')
fig.show()


In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

# Assuming 'data' is your DataFrame containing the columns
# Replace 'data' with your actual DataFrame name if different
columns_to_correlate = ['PMS', 'PVS', 'VMS', 'SAMX', 'signal', 'SAMX_signal', 'PVS_signal', 'VMS_signal']

# Calculate the correlation matrix
correlation_matrix = df[columns_to_correlate].corr()

# Create a heatmap
plt.figure(figsize=(12, 6))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', vmin=-1, vmax=1, center=0, square=True, fmt='.2f')
plt.title('Correlation Heatmap of PMS, PVS, VMS, SAMX, and Signals')
plt.xticks(rotation=45)
plt.yticks(rotation=0)
plt.tight_layout()

# Display the plot
plt.show()

In [ ]:
import pandas as pd
import plotly.express as px

# Assuming 'df' is your DataFrame with the required columns
# Replace 'df' with your actual DataFrame name if different
indicators = ['PMS', 'PVS', 'VMS', 'SAMX']
signal_column = 'signal'

# Ensure the signal column contains only 0, 1, 2
df[signal_column] = df[signal_column].astype(int).clip(1, 2)  # Clip to ensure values are 0, 1, or 2

# Create interactive box plots for each indicator vs. signal
for indicator in indicators:
    fig = px.box(df, x=signal_column, y=indicator,
                 title=f'{indicator} Distribution by Signal',
                 labels={'signal': 'Signal (0=Neutral, 1=Buy, 2=Sell)', indicator: indicator},
                 color=signal_column,
                 points="all")  # Show all points including outliers

    # Customize layout
    fig.update_layout(showlegend=True,
                      template='plotly_dark',
                      height=500,
                      width=800)

    # Add median annotations
    medians = df.groupby(signal_column)[indicator].median()
    for i, median in enumerate(medians):
        fig.add_annotation(x=i, y=median,
                           text=f'Median: {median:.2f}',
                           showarrow=True,
                           arrowhead=1,
                           ax=0,
                           ay=-40)

    # Show the plot
    fig.show()

# Optional: Print summary statistics for further analysis
for indicator in indicators:
    print(f"\nSummary Statistics for {indicator} by Signal:")
    print(df.groupby(signal_column)[indicator].describe())

In [ ]:
df1['Year'] = df1['Entry Time'].dt.year

fig = px.box(
    df1,
    x='Year',
    y='PnL',
    title='📦 PnL Distribution by Year',
    points='all',
    template='plotly_dark'
)
fig.show()


In [ ]:
df1['Win'] = df1['PnL'] > 0

winrate_by_year = df1.groupby('Year')['Win'].mean().reset_index()

fig = px.line(
    winrate_by_year,
    x='Year',
    y='Win',
    title='✅ Win Rate Over Years',
    markers=True,
    template='plotly_dark'
)
fig.update_yaxes(title='Win Rate (%)', tickformat='.0%')
fig.show()


In [ ]:
fig = px.histogram(
    df1,
    x='Exit Reason',
    color='Direction',
    barmode='group',
    title='🚪 Exit Reasons Breakdown (Long vs Short)',
    template='plotly_dark'
)
fig.show()


In [ ]:
df1['Exit Time'] = pd.to_datetime(df1['Exit Time'])
df1['Hold Mins'] = (df1['Exit Time'] - df1['Entry Time']).dt.total_seconds() / 60

fig = px.scatter(
    df1,
    x='Hold Mins',
    y='PnL',
    color='Direction',
    title='⏱️ PnL vs Holding Time',
    template='plotly_dark'
)
fig.show()


In [ ]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

# Convert dates
for col in ['Entry Time', 'Exit Time']:
    df1[col] = pd.to_datetime(df1[col])
df['Date'] = pd.to_datetime(df['Date'])

# 1. Rolling PnL + Win Rate Analysis
df1['date'] = df1['Entry Time'].dt.to_period('Q').dt.to_timestamp()
grouped = df1.groupby('date')

roll_pnl = grouped['PnL'].sum().reset_index(name='Quarterly PnL')
win_rate = grouped.apply(lambda x: (x['PnL'] > 0).mean()).reset_index(name='Win Rate')

fig1 = px.line(roll_pnl, x='date', y='Quarterly PnL', title='📉 Rolling Quarterly PnL')
fig2 = px.line(win_rate, x='date', y='Win Rate', title='✅ Win Rate per Quarter')

fig1.show()
fig2.show()

# 2. Signal Strength vs PnL Mapping
df_merged = pd.merge_asof(df1.sort_values('Entry Time'),
                          df.sort_values('Date'),
                          left_on='Entry Time',
                          right_on='Date',
                          by='Ticker')

for score in ['PVS', 'VMS', 'SAMX']:
    df_merged['bucket'] = pd.cut(df_merged[score], bins=10)
    df_merged['bucket'] = df_merged['bucket'].astype(str)
    score_pnl = df_merged.groupby('bucket', observed=False)['PnL'].mean().reset_index()
    fig = px.bar(score_pnl, x='bucket', y='PnL', title=f'🎯 Avg PnL vs {score} Score Buckets')
    fig.show()

# 3. Volatility Regimes vs PnL
df_merged['ATR_bucket'] = pd.cut(df_merged['ATR'], bins=10)
df_merged['ATR_bucket'] = df_merged['ATR_bucket'].astype(str)  # Convert Interval to string
atrs = df_merged.groupby('ATR_bucket', observed=False)['PnL'].mean().reset_index()
fig = px.bar(atrs, x='ATR_bucket', y='PnL', title='📊 Avg PnL by ATR Volatility Regime')
fig.show()

# 4. Rolling Sharpe Ratio
def sharpe(x):
    return x.mean() / (x.std() + 1e-6)

sharpe_by_q = df1.groupby('date')['PnL'].apply(sharpe).reset_index(name='Sharpe')
fig = px.line(sharpe_by_q, x='date', y='Sharpe', title='📉 Rolling Sharpe Ratio by Quarter')
fig.show()

# 5. XGBoost Predictive Edge Modeling
features = ['PVS', 'VMS', 'SAMX', 'ATR', 'PMS']
df_merged['success'] = (df_merged['PnL'] > 0).astype(int)

X = df_merged[features]
y = df_merged['success']

X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.3)

model = GradientBoostingClassifier().fit(X_train, y_train)
y_pred = model.predict_proba(X_test)[:, 1]
auc = roc_auc_score(y_test, y_pred)
print(f"🎯 Predictive Power (AUC): {auc:.3f}")

# 6. Trade Duration and Drawdown
df1['duration'] = (df1['Exit Time'] - df1['Entry Time']).dt.total_seconds() / 60
duration_trend = df1.groupby('date')['duration'].mean().reset_index()
fig = px.line(duration_trend, x='date', y='duration', title='⏱ Avg Trade Duration per Quarter')
fig.show()

# Optional: Drawdown visualization
cum_pnl = df1.groupby('date')['PnL'].sum().cumsum()
drawdown = cum_pnl - cum_pnl.cummax()
fig = px.area(x=cum_pnl.index, y=drawdown, title='🔻 Strategy Drawdown Over Time')
fig.update_traces(fill='tozeroy')
fig.show()